# Hospital Admission Forecasting
## Forecasting Daily Hospital Admissions for Capacity Planning

**Notebook #12 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Hospital Admissions Data (`ashishsahani/hospital-admissions-data`) |
| **Target** | Daily admission count |
| **Frequency** | Daily (`D`) |
| **Primary TS Library** | StatsForecast (AutoARIMA + AutoETS + AutoTheta) |

## Learning Objectives
1. Forecast daily hospital admissions to support bed and staffing capacity planning
2. Identify weekly and seasonal patterns in admission demand
3. Understand why healthcare forecasting is different from retail forecasting
4. Apply proper time-aware splits respecting the clinical context
5. Discuss the operational consequences of over- vs under-forecasting admissions

## Problem Statement
Hospital administrators need advance notice of demand to:
- **Allocate beds** across wards (general, surgical, ICU)
- **Schedule nursing staff** and residents
- **Prepare operating room slots**

A 14-day horizon is operationally the most useful window — it aligns with
shift-scheduling and elective-procedure planning cycles.

> Goal: Forecast daily admissions 14 days ahead using historical admission records.

## Why Healthcare Forecasting Is Unique
Unlike retail demand, hospital admissions have:
- **Inelastic demand** — patients don't delay genuine emergencies
- **Weekly structure** — elective admissions drop sharply on weekends
- **Seasonal surges** — winter respiratory illness, summer trauma
- **Event-driven spikes** — public health events, flu season
- **No negative demand** — admissions are always ≥ 0
- **Autocorrelation patterns** — long-stay patients inflate next-day census

## Dataset Overview
**Source:** Kaggle — `ashishsahani/hospital-admissions-data`

**License:** Check Kaggle page for licensing details.

**Expected columns**: Date/timestamp column + admission count column.
The notebook auto-detects them with a fallback to column-inspection.

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = (a != 0)
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<40s}  MAE={mae:>9,.2f}  RMSE={rmse:>9,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}

print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG  = "ashishsahani/hospital-admissions-data"
FREQ         = "D"
SEASON_LEN   = 7
HORIZON      = 14
TEST_DAYS    = 14
VAL_DAYS     = 14
FLAML_BUDGET = 90
RANDOM_STATE = 42
print("Config OK.")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists():
    print(f"[OK] kaggle.json at {_j}"); _ok = True
if not _ok:
    print("WARNING: Kaggle credentials missing.")
    print("  Option A: set KAGGLE_USERNAME + KAGGLE_KEY env vars")
    print("  Option B: place kaggle.json at ~/.kaggle/kaggle.json")
    raise EnvironmentError("Set Kaggle credentials and restart.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files = sorted(data_path.rglob("*.csv"))
print([f.name for f in csv_files])
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Data Validation & Column Detection
We auto-detect the date and admission-count columns.

In [ ]:
# Auto-detect date column
date_col = None
for col in df.columns:
    try:
        pd.to_datetime(df[col].dropna().iloc[:5])
        date_col = col
        break
    except Exception:
        continue
if date_col is None:
    # Try common names
    for c in df.columns:
        if any(kw in c.lower() for kw in ["date","time","day"]):
            date_col = c; break
print(f"Date column detected: {date_col}")

# Auto-detect admission count column
count_col = None
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols:
    if any(kw in c.lower() for kw in ["admit","admission","count","total","visit","patient"]):
        count_col = c; break
if count_col is None and len(num_cols) > 0:
    count_col = num_cols[0]
print(f"Count column detected: {count_col}")

In [ ]:
df["_date"] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=["_date"]).sort_values("_date").reset_index(drop=True)
print(f"Date range: {df['_date'].min().date()} → {df['_date'].max().date()}")
print(f"Rows: {len(df):,}")
print(f"\nCount column stats:")
print(df[count_col].describe())

## Build Daily Admission Series

In [ ]:
daily_raw = df.groupby(df["_date"].dt.date)[count_col].sum().reset_index()
daily_raw.columns = ["ds","y"]
daily_raw["ds"] = pd.to_datetime(daily_raw["ds"])
daily_raw = daily_raw.sort_values("ds").reset_index(drop=True)
full_idx = pd.date_range(daily_raw["ds"].min(), daily_raw["ds"].max(), freq="D")
daily = daily_raw.set_index("ds").reindex(full_idx, fill_value=0).reset_index()
daily.columns = ["ds","y"]
print(f"Series: {len(daily)} days  |  Total admissions: {daily['y'].sum():,}")
print(daily["y"].describe().round(1))

## EDA

In [ ]:
fig = px.line(daily, x="ds", y="y",
              title="Daily Hospital Admissions",
              labels={"ds":"Date","y":"Admissions"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
daily["dow"] = daily["ds"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig = px.box(daily[daily["y"]>0], x="dow", y="y",
             category_orders={"dow":dow_order},
             title="Admissions by Day of Week",
             labels={"dow":"Day","y":"Admissions"})
fig.update_layout(template="plotly_white")
fig.show()
print("Typically: elective admissions lower on weekends; emergency admissions more uniform.")

In [ ]:
if len(daily) > 3*SEASON_LEN:
    ts_d = daily.set_index("ds")["y"].asfreq("D").interpolate()
    decomp = seasonal_decompose(ts_d, model="additive", period=SEASON_LEN)
    fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0],title="Observed")
    decomp.trend.plot(ax=axes[1],title="Trend")
    decomp.seasonal.plot(ax=axes[2],title="Seasonal (weekly)")
    decomp.resid.plot(ax=axes[3],title="Residual")
    plt.tight_layout(); plt.show()

In [ ]:
adf = adfuller(daily["y"].dropna())
print(f"ADF p={adf[1]:.4f} → {'Stationary' if adf[1]<0.05 else 'Non-stationary'}")
fig,axes = plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"],  lags=min(60,len(daily)//3), ax=axes[0])
plot_pacf(daily["y"], lags=min(60,len(daily)//3), ax=axes[1])
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
y = daily["y"]
print(f"Mean={y.mean():.1f}  Std={y.std():.1f}  CV={y.std()/y.mean():.3f}")
zero_pct = (y==0).mean()*100
print(f"Zero-admission days: {(y==0).sum()} ({zero_pct:.1f}%)")
if zero_pct > 20:
    print("WARNING: High zero rate may indicate data collection gaps, not true zero-demand days.")

## Train / Validation / Test Split

In [ ]:
n=len(daily); test_start=n-TEST_DAYS; val_start=test_start-VAL_DAYS
ts_train=daily.iloc[:val_start].copy(); ts_val=daily.iloc[val_start:test_start].copy()
ts_test=daily.iloc[test_start:].copy(); ts_trainval=daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max()<ts_val["ds"].min()
assert ts_val["ds"].max()<ts_test["ds"].min()
print("No overlap confirmed.")

## Preprocessing

In [ ]:
print(f"NaN in y: {daily['y'].isna().sum()}")
# A value of 0 on a weekday may be a data gap — flag but don't impute
weekday_zeros = daily[(daily['y']==0) & (daily['ds'].dt.dayofweek<5)]
print(f"Weekday zero-admission days: {len(weekday_zeros)} (potential data gaps)")

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df=df_in.copy().sort_values("ds").reset_index(drop=True); y=df["y"]
    for lag in [1,7,14,28]: df[f"lag_{lag}"]=y.shift(lag)
    for w in [7,14]:
        df[f"rmean_{w}"]=y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]=y.shift(1).rolling(w).std()
    df["dow"]=df["ds"].dt.dayofweek
    df["is_weekend"]=(df["dow"]>=5).astype(int)
    df["month"]=df["ds"].dt.month; df["quarter"]=df["ds"].dt.quarter
    return df
full_feat=build_feats(daily)
feat_train=full_feat.iloc[:val_start].dropna().copy()
feat_val=full_feat.iloc[val_start:test_start].dropna().copy()
feat_test=full_feat.iloc[test_start:].dropna().copy()
feat_trainval=full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS=[c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features: {FEAT_COLS}")

## Baselines

In [ ]:
results=[]; y_test=ts_test["y"].values; last_tv=ts_trainval["y"].values
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-1]),"Naive"))
sn=np.tile(last_tv[-SEASON_LEN:],TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test,sn,"Seasonal Naive (7-day)"))
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-7:].mean()),"MA-7"))

## LazyPredict

In [ ]:
try:
    lz=LazyRegressor(verbose=0,ignore_warnings=True)
    lm,_=lz.fit(feat_train[FEAT_COLS],feat_val[FEAT_COLS],feat_train["y"],feat_val["y"])
    print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}"); lm=None

## FLAML AutoML

In [ ]:
flaml=AutoML()
flaml.fit(feat_trainval[FEAT_COLS],feat_trainval["y"],task="regression",metric="rmse",
          time_budget=FLAML_BUDGET,verbose=0,seed=RANDOM_STATE)
print(f"Best: {flaml.best_estimator}")
flaml_pred = flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values,flaml_pred,f"FLAML ({flaml.best_estimator})"))

## StatsForecast

In [ ]:
sf_df=ts_trainval[["ds","y"]].assign(unique_id="hospital")
sf=StatsForecast(models=[AutoARIMA(season_length=SEASON_LEN),AutoETS(season_length=SEASON_LEN),
                          AutoTheta(season_length=SEASON_LEN),SeasonalNaive(season_length=SEASON_LEN)],
                 freq=FREQ,n_jobs=1)
sf.fit(sf_df); sf_fc=sf.forecast(h=HORIZON)
for col in ["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive"]:
    if col in sf_fc.columns:
        results.append(metrics(y_test,sf_fc[col].values[:TEST_DAYS],f"SF-{col}"))

## MLForecast

In [ ]:
mlf_df=ts_trainval[["ds","y"]].assign(unique_id="hospital")
mlf=MLForecast(models={"lgbm":LGBMRegressor(n_estimators=200,learning_rate=0.05,
                                              verbosity=-1,random_state=RANDOM_STATE)},
               freq=FREQ,lags=[1,7,14],lag_transforms={1:[("rolling_mean",7)]},
               date_features=["dayofweek","month"],num_threads=2)
mlf.fit(mlf_df); mlf_fc=mlf.predict(h=HORIZON)
results.append(metrics(y_test,mlf_fc["lgbm"].values[:TEST_DAYS],"MLF-LightGBM"))

## Top 3 & Comparison

In [ ]:
res_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3=res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig=px.bar(res_df,x="Model",y="RMSE",color="RMSE",color_continuous_scale="RdYlGn_r",
           title="Hospital Admission Forecast — Model Comparison")
fig.update_layout(template="plotly_white",xaxis_tickangle=-35); fig.show()

## Forecast Plots

In [ ]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"].tail(60),y=ts_trainval["y"].tail(60),
    name="Train",line=dict(color="royalblue",dash="dot",width=1)))
fig.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],
    name="Actual",line=dict(color="black",width=2)))
preds={}
if flaml_pred is not None: preds[f"FLAML ({flaml.best_estimator})"]=flaml_pred
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in sf_fc.columns: preds[f"SF-{col}"]=sf_fc[col].values[:TEST_DAYS]
if "lgbm" in mlf_fc.columns: preds["MLF-LightGBM"]=mlf_fc["lgbm"].values[:TEST_DAYS]
preds["Seasonal Naive (7-day)"]=sn
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in preds:
        fig.add_trace(go.Scatter(x=ts_test["ds"][:len(preds[m])],y=preds[m],
            name=f"#{i+1} {m}",line=dict(color=colors[i],width=2)))
fig.update_layout(title="Top 3 Forecasts vs Actual — Daily Admissions",
                  template="plotly_white",xaxis_title="Date",yaxis_title="Admissions")
fig.show()

## Error Analysis

In [ ]:
best_m=top3.iloc[0]["Model"]
if best_m in preds:
    bp=np.asarray(preds[best_m]).ravel(); ba=y_test[:len(bp)]; err=ba-bp
    print(f"Bias={err.mean():+.2f}  Std={err.std():.2f}")
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    axes[0].hist(err,bins=20,color="steelblue",edgecolor="white")
    axes[0].axvline(0,color="red",linestyle="--"); axes[0].set_title("Error Distribution")
    axes[1].scatter(ba,bp,s=50,alpha=0.7,color="steelblue")
    lim=max(ba.max(),bp.max())*1.1; axes[1].plot([0,lim],[0,lim],"r--")
    axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
    axes[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Interpretation & Insights
1. **Weekend dip** is the most consistent pattern — elective procedures pause on Saturday/Sunday
2. **AutoARIMA** is a strong baseline here as it handles differencing and seasonal ARIMA orders automatically
3. **Under-forecasting is riskier than over-forecasting** in healthcare — a short-staffed ward
   leads to patient harm; over-staffing is merely inefficient
4. **Seasonal patterns in respiratory illness** (winter spikes) are forecastable with monthly features

## Limitations
1. No case-mix information — a day with 50 trauma admissions ≠ 50 elective admissions
2. No census accounting — long-stay patients hold beds without being admitted
3. Missing seasonal covariates (flu season, temperature, holiday proximity)
4. Single-hospital — multi-site forecasts require hierarchy reconciliation

## How to Improve
1. Add flu-season index (CDC ILI data) as weekly covariate
2. Separate emergency vs elective admission series (different patterns)
3. Add a "census forecast" layer: census = prior_census + admissions - discharges
4. Use TFT (Temporal Fusion Transformer) for multi-covariate long-horizon forecasting

## Production Considerations
| Concern | Approach |
|---------|---------|
| Retraining | Weekly — new admissions data available daily |
| Safety buffer | Forecast + 1σ for staffing decisions (conservative) |
| Elective / emergency split | Two separate models, reconciled at ward level |
| Integration | Push forecast to nurse-scheduling system at T-14 days |

## Common Mistakes
1. Treating zero-admission days as missing data (clipping to 0 then interpolating destroys the signal)
2. Averaging across hospitals with very different profiles
3. Forecasting census instead of admissions (different metric, different model)
4. Ignoring that bank-holiday demand is different from regular-Sunday demand

## Exercises
1. Separate weekend vs weekday series and compare forecast accuracy
2. Add a winter indicator (Nov–Feb) as a binary feature and measure lift
3. Try a 7-day rolling forecast (walk-forward validation) over 6 weeks
4. Compare AutoARIMA vs Theta on this specific dataset — which wins and why?

## Summary
- Built daily admission time series from a hospital admissions dataset
- EDA: strong weekly seasonality, possible seasonal trend
- Five approaches compared; top-3 selected by test RMSE
- Healthcare context: under-forecasting more dangerous than over-forecasting

---
*Notebook #12 of 50 — Time Series Forecasting Portfolio*